In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, to_timestamp, round
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

spark = SparkSession.builder.appName("Silver_Transformations").getOrCreate()

#Step 1: Read from Bronze table 
print("Reading from Bronze layer...")
bronze_df = spark.read.format("delta").table("bronze_payments")
print(f"Bronze row count (with duplicates): {bronze_df.count()}")

#Step 2: Remove duplicates
#Keep only the first occurrence of each payment_id
#This handles Stripe's at-least-once delivery problem
window = Window.partitionBy("payment_id").orderBy("timestamp")
deduped_df = bronze_df.withColumn("row_num", row_number().over(window)) \
                      .filter("row_num = 1") \
                      .drop("row_num")

print(f"Row count after deduplication: {deduped_df.count()}")

#Step 3: Fix data types
#Convert timestamp string to proper timestamp type
#Round amount to 2 decimal places
cleaned_df = deduped_df \
    .withColumn("timestamp", to_timestamp("timestamp")) \
    .withColumn("amount", round("amount", 2))

#Step4: Update 1 layer marker 
silver_df = cleaned_df \
    .withColumn("processed_timestamp", current_timestamp()) \
    .withColumn("layer", lit("silver"))

#Step 5: Show the cleaned data
print("\nSilver layer - cleaned payment events:")
silver_df.show(truncate=False)

#Step 6: Save to Delta Lake as Silver table 
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_payments")

print("Silver table saved successfully!")
print(f"Total clean records: {silver_df.count()}")

Reading from Bronze layer...
Bronze row count (with duplicates): 6
Row count after deduplication: 5

Silver layer - cleaned payment events:
+------------+------------+-----------+------+--------+--------------+---------+-------------------+--------------+--------------------------+------+--------------------------+
|payment_id  |user_id     |merchant_id|amount|currency|payment_method|status   |timestamp          |source_system |ingest_timestamp          |layer |processed_timestamp       |
+------------+------------+-----------+------+--------+--------------+---------+-------------------+--------------+--------------------------+------+--------------------------+
|pay_a3f9b2c1|usr_7829kdf9|mer_4421abc|149.99|USD     |credit_card   |succeeded|2024-01-15 14:32:11|mobile_ios    |2026-04-12 19:27:26.127284|silver|2026-04-13 03:47:02.784646|
|pay_b7d2e4f1|usr_3312xyz |mer_8821def|23.5  |USD     |debit_card    |failed   |2024-01-15 14:32:12|web_checkout  |2026-04-12 19:27:26.127284|silver|202